<a href="https://colab.research.google.com/github/Shivam250124/Machine_Learning-/blob/main/Model_Evaluation_grid_search_cv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h3 align="center">Grid Search CV</h3>

We will generate a synthetic dataset

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_features=10,
    n_samples=1000,
    n_informative=8,
    n_redundant=2,
    n_repeated=0,
    n_classes=2,
    random_state=42
)

### Method 1: Evaluate the model using train, test split and tune parameters by trial and error

In [3]:
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = DecisionTreeClassifier(criterion="entropy", max_depth=10) # criteria: "gini" or "entropy", max_depth=5 or 10
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.84      0.76      0.80       130
           1       0.77      0.84      0.80       120

    accuracy                           0.80       250
   macro avg       0.80      0.80      0.80       250
weighted avg       0.80      0.80      0.80       250



### Method 2: Use K Fold Cross Validation

In [4]:
from sklearn.model_selection import cross_val_score

cross_val_score(DecisionTreeClassifier(criterion="gini", max_depth=5), X, y, cv=5)

array([0.78 , 0.81 , 0.75 , 0.795, 0.775])

In [5]:
cross_val_score(DecisionTreeClassifier(criterion="entropy", max_depth=5), X, y, cv=5)

array([0.765, 0.785, 0.745, 0.81 , 0.78 ])

In [6]:
criterion = ["gini", "entropy"]
max_depth = [5, 10, 15]

avg_scores = {}

for c in criterion:
    for d in max_depth:
        clf = DecisionTreeClassifier(criterion=c, max_depth=d)
        score_list = cross_val_score(clf, X, y, cv=5)#When you supply int parameter in cv and if the estimater is a classifier,
        #it will be default use Stratified K Fold where k is set to the number that you specified for cv
        avg_scores[c + "_" + str(d)] = np.average(score_list)

avg_scores

{'gini_5': np.float64(0.7819999999999999),
 'gini_10': np.float64(0.788),
 'gini_15': np.float64(0.79),
 'entropy_5': np.float64(0.78),
 'entropy_10': np.float64(0.787),
 'entropy_15': np.float64(0.817)}

### Method 3: Use GridSearchCV

In [7]:
from sklearn.model_selection import GridSearchCV

clf = GridSearchCV(
    DecisionTreeClassifier(),
    {'criterion': ["gini", "entropy"],
     'max_depth': [5, 10, 15]
     },
    cv=5,
    return_train_score=False
)
clf.fit(X, y)
clf.cv_results_

{'mean_fit_time': array([0.00524449, 0.00729356, 0.00766048, 0.00826283, 0.01315522,
        0.01353197]),
 'std_fit_time': array([0.00032407, 0.00053255, 0.00057334, 0.00178142, 0.00094396,
        0.00121567]),
 'mean_score_time': array([0.00053096, 0.00054688, 0.00054302, 0.0006803 , 0.00085754,
        0.00086527]),
 'std_score_time': array([4.57718966e-05, 1.46128612e-05, 2.82296458e-05, 1.46943125e-04,
        9.84515047e-05, 1.67298193e-04]),
 'param_criterion': masked_array(data=['gini', 'gini', 'gini', 'entropy', 'entropy',
                    'entropy'],
              mask=[False, False, False, False, False, False],
        fill_value=np.str_('?'),
             dtype=object),
 'param_max_depth': masked_array(data=[5, 10, 15, 5, 10, 15],
              mask=[False, False, False, False, False, False],
        fill_value=999999),
 'params': [{'criterion': 'gini', 'max_depth': 5},
  {'criterion': 'gini', 'max_depth': 10},
  {'criterion': 'gini', 'max_depth': 15},
  {'criterion': '

In [8]:
df = pd.DataFrame(clf.cv_results_)
df.head(3)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_depth,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.005244,0.000324,0.000531,0.000046,gini,5,"{'criterion': 'gini', 'max_depth': 5}",0.775,0.795,0.755,0.800,0.775,0.780,0.016125,6
1,0.007294,0.000533,0.000547,0.000015,gini,10,"{'criterion': 'gini', 'max_depth': 10}",0.795,0.745,0.815,0.785,0.815,0.791,0.025768,2
2,0.007660,0.000573,0.000543,0.000028,gini,15,"{'criterion': 'gini', 'max_depth': 15}",0.820,0.735,0.785,0.780,0.810,0.786,0.029563,4


In [9]:
df[["param_criterion", "param_max_depth", "mean_test_score"]]

,param_criterion,param_max_depth,mean_test_score
0,gini,5,0.780
1,gini,10,0.791
2,gini,15,0.786
3,entropy,5,0.781
4,entropy,10,0.790
5,entropy,15,0.809


In [10]:
clf.best_params_

{'criterion': 'entropy', 'max_depth': 15}

In [11]:
model = clf.best_estimator_
model

DecisionTreeClassifier(criterion='entropy', max_depth=15)

### Now let's try different models with different parameters

In [13]:
from sklearn import svm

model_params = {
    'decision_tree': {
        'model': DecisionTreeClassifier(),
        'params' : {
            'criterion': ["gini", "entropy"],
            'max_depth': [5, 10, 15]
        }
    },
    'svm': {
        'model': svm.SVC(gamma='auto'),
        'params' : {
            'C': [1,10,20],
            'kernel': ['rbf','linear']
        }
    }
}

scores = []

for key, val in model_params.items():
    clf = GridSearchCV(
        val['model'],
        val['params'],
        cv=5,
        return_train_score=False
    )
    clf.fit(X, y)
    scores.append({
        'model': key,
        'best_score': clf.best_score_,
        'best_params': clf.best_params_
    })

scores

[{'model': 'decision_tree',
  'best_score': np.float64(0.8140000000000001),
  'best_params': {'criterion': 'entropy', 'max_depth': 15}},
 {'model': 'svm',
  'best_score': np.float64(0.9260000000000002),
  'best_params': {'C': 1, 'kernel': 'rbf'}}]

In [14]:
pd.DataFrame(scores)

,model,best_score,best_params
0,decision_tree,0.814,"{'criterion': 'entropy', 'max_depth': 15}"
1,svm,0.926,"{'C': 1, 'kernel': 'rbf'}"


In [15]:
df = pd.DataFrame(scores, columns=["model", "best_score", "best_params"])
df

,model,best_score,best_params
0,decision_tree,0.814,"{'criterion': 'entropy', 'max_depth': 15}"
1,svm,0.926,"{'C': 1, 'kernel': 'rbf'}"
